In [1]:
class TuringMachine:
    def __init__(self, rules, start_state="A", blank=0):
        """
        rules: dict of the form:
            (state, symbol) -> (new_symbol, move, next_state)
        move: -1 for left, +1 for right
        """
        self.rules = rules
        self.state = start_state
        self.blank = blank
        self.tape = {0: blank}
        self.head = 0
        self.steps = 0

    def read(self):
        return self.tape.get(self.head, self.blank)

    def write(self, symbol):
        self.tape[self.head] = symbol

    def move(self, direction):
        self.head += direction

    def step(self):
        symbol = self.read()
        key = (self.state, symbol)

        if key not in self.rules:
            # No rule → halt
            return False

        new_symbol, direction, next_state = self.rules[key]
        self.write(new_symbol)
        self.move(direction)
        self.state = next_state
        self.steps += 1
        return True

    def run(self, max_steps=10_000_000):
        while self.steps < max_steps:
            if not self.step():
                return True  # halted
        return False  # exceeded max steps (likely non-halting)

In [2]:
rules = {
    ("A", 0): (1, +1, "B"),
    ("A", 1): (1, -1, "B"),
    ("B", 0): (1, -1, "A"),
    ("B", 1): (1, +1, "HALT")
}

tm = TuringMachine(rules)
halted = tm.run()

print("Halted:", halted)
print("Steps:", tm.steps)
print("Tape:", tm.tape)

Halted: True
Steps: 6
Tape: {0: 1, 1: 1, -1: 1, -2: 1}


In [4]:
rules = {
    ("A", 0): (1, +1, "B"),
    ("A", 1): (1, -1, "C"),
    ("B", 0): (1, -1, "A"),
    ("B", 1): (1, +1, "B"),
    ("C", 0): (1, -1, "B"),
    ("C", 1): (1, -1, "HALT")
}

tm = TuringMachine(rules)
tm.run()

print("Halted:", halted)
print("Steps:", tm.steps)
print("Tape:", tm.tape)

Halted: True
Steps: 13
Tape: {0: 1, 1: 1, -1: 1, -2: 1, -3: 1, 2: 1}


In [5]:
rules = {
    ("A", 0): (1, +1, "B"),
    ("A", 1): (1, -1, "C"),

    ("B", 0): (1, +1, "C"),
    ("B", 1): (1, +1, "B"),

    ("C", 0): (1, +1, "D"),
    ("C", 1): (0, -1, "A"),

    ("D", 0): (1, -1, "A"),
    ("D", 1): (1, +1, "HALT"),
}

tm = TuringMachine(rules)
halted = tm.run(max_steps=1_000_000)

print("Halted:", halted)
print("Steps:", tm.steps)
print("Non-blank cells:", sum(v for v in tm.tape.values()))

Halted: True
Steps: 9
Non-blank cells: 4


In [6]:
class FastTM:
    def __init__(self, rules, start="A", blank=0):
        self.rules = rules
        self.state = start
        self.blank = blank
        self.tape = {}
        self.head = 0
        self.steps = 0

    def run(self, max_steps=100_000_000):
        tape = self.tape
        head = self.head
        state = self.state
        blank = self.blank
        rules = self.rules

        while self.steps < max_steps:
            symbol = tape.get(head, blank)
            key = (state, symbol)

            if key not in rules:
                # halt
                self.tape = tape
                self.head = head
                self.state = state
                return True

            new_symbol, move, next_state = rules[key]
            tape[head] = new_symbol
            head += move
            state = next_state
            self.steps += 1

        self.tape = tape
        self.head = head
        self.state = state
        return False

In [7]:
rules_5bb = {
    ("A", 0): (1, +1, "B"),
    ("A", 1): (1, -1, "C"),

    ("B", 0): (1, +1, "C"),
    ("B", 1): (1, +1, "B"),

    ("C", 0): (1, +1, "D"),
    ("C", 1): (0, -1, "E"),

    ("D", 0): (1, -1, "A"),
    ("D", 1): (1, -1, "D"),

    ("E", 0): (1, +1, "HALT"),
    ("E", 1): (0, -1, "A"),
}

tm = FastTM(rules_5bb)
halted = tm.run(max_steps=100_000_000)

print("Halted:", halted)
print("Steps:", tm.steps)
print("Number of 1s:", sum(1 for v in tm.tape.values() if v == 1))

Halted: True
Steps: 47176870
Number of 1s: 4098
